In [ ]:
# ============================================
# Import Required Libraries
# Run This Block Only Once
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Train-Test Split & Cross Validation
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict
)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Models
from sklearn.ensemble import RandomForestClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

print("All libraries imported successfully.")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Load dataset
df = pd.read_csv("term-deposit-marketing-2020.csv")

# Basic preview
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

In [2]:
# Dataset information
print("Dataset Information:")
df.info()

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        40000 non-null  int64 
 1   job        40000 non-null  object
 2   marital    40000 non-null  object
 3   education  40000 non-null  object
 4   default    40000 non-null  object
 5   balance    40000 non-null  int64 
 6   housing    40000 non-null  object
 7   loan       40000 non-null  object
 8   contact    40000 non-null  object
 9   day        40000 non-null  int64 
 10  month      40000 non-null  object
 11  duration   40000 non-null  int64 
 12  campaign   40000 non-null  int64 
 13  y          40000 non-null  object
dtypes: int64(5), object(9)
memory usage: 4.3+ MB

Missing Values:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0

In [29]:

# Check Unique Values in Each Column

for column in df.columns:
    print(f"\n{'='*50}")
    print(f"Column: {column}")
    print(f"Total Unique Values: {df[column].nunique()}")
    print(f"Unique Values: {df[column].unique()}")


Column: age
Total Unique Values: 70
Unique Values: [58 44 33 47 35 28 42 43 41 29 53 57 51 45 60 56 32 25 40 39 52 46 36 49
 59 37 50 54 55 48 24 38 31 30 27 34 23 26 61 22 21 20 66 62 83 75 67 70
 65 68 64 69 72 71 19 76 85 63 90 82 73 74 78 80 94 79 77 86 95 81]

Column: job
Total Unique Values: 12
Unique Values: ['management' 'technician' 'entrepreneur' 'blue-collar' 'unknown'
 'retired' 'admin' 'services' 'self-employed' 'unemployed' 'housemaid'
 'student']

Column: marital
Total Unique Values: 3
Unique Values: ['married' 'single' 'divorced']

Column: education
Total Unique Values: 4
Unique Values: ['tertiary' 'secondary' 'unknown' 'primary']

Column: default
Total Unique Values: 2
Unique Values: ['no' 'yes']

Column: balance
Total Unique Values: 6849
Unique Values: [  2143     29      2 ...   7222   3402 102127]

Column: housing
Total Unique Values: 2
Unique Values: ['yes' 'no']

Column: loan
Total Unique Values: 2
Unique Values: ['no' 'yes']

Column: contact
Total Unique Values:

In [32]:
df_model = df.copy()

# Convert Age into Age Groups
df_model["age_group"] = pd.cut(
    df_model["age"],
    bins=[0, 30, 40, 50, 60, np.inf],
    labels=["18-30", "31-40", "41-50", "51-60", "60+"]
)

# Convert Day into Week Groups
df_model["week"] = pd.cut(
    df_model["day"],
    bins=[0, 7, 14, 21, 31],
    labels=[1, 2, 3, 4]
).astype("int64")

# Convert Month into Quarter
quarter_mapping = {
    "jan": 1, "feb": 1, "mar": 1,
    "apr": 2, "may": 2, "jun": 2,
    "jul": 3, "aug": 3, "sep": 3,
    "oct": 4, "nov": 4, "dec": 4
}

df_model["quarter"] = df_model["month"].map(quarter_mapping)


# Convert Yes/No Columns into 0/1

binary_columns = [
    "default",
    "housing",
    "loan",
    "y"
]

for column in binary_columns:
    df_model[column] = (
        df_model[column]
        .map({"no": 0, "yes": 1})
        .astype("int64")
    )

# Remove original and excluded features
df_model.drop(
    columns=[
        "age",
        "day",
        "month",
        ],
    inplace=True
)

# Check final modeling columns
print("Modeling Dataset Shape:", df_model.shape)

print("\nFinal Columns:")
print(df_model.columns.tolist())

print("\nFirst 5 Rows:")
display(df_model.head())

Modeling Dataset Shape: (40000, 14)

Final Columns:
['job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'duration', 'campaign', 'y', 'age_group', 'week', 'quarter']

First 5 Rows:


,job,marital,education,default,balance,housing,loan,contact,duration,campaign,y,age_group,week,quarter
0,management,married,tertiary,0,2143,1,0,unknown,261,1,0,51-60,1,2
1,technician,single,secondary,0,29,1,0,unknown,151,1,0,41-50,1,2
2,entrepreneur,married,secondary,0,2,1,1,unknown,76,1,0,31-40,1,2
3,blue-collar,married,unknown,0,1506,1,0,unknown,92,1,0,41-50,1,2
4,unknown,single,unknown,0,1,0,0,unknown,198,1,0,31-40,1,2


In [33]:
df_model.dtypes

job            object
marital        object
education      object
default         int64
balance         int64
housing         int64
loan            int64
contact        object
duration        int64
campaign        int64
y               int64
age_group    category
week            int64
quarter         int64
dtype: object

## Train-Test Split

In [34]:
from sklearn.model_selection import train_test_split

X = df_model.drop(
    columns=['y', 'contact']
)

y = df_model['y']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (30000, 12)
X_test shape : (10000, 12)
y_train shape: (30000,)
y_test shape : (10000,)


In [35]:
print("X_train dtypes:")
print(X_train.dtypes)

print("\nX_test dtypes:")
print(X_test.dtypes)

X_train dtypes:
job            object
marital        object
education      object
default         int64
balance         int64
housing         int64
loan            int64
duration        int64
campaign        int64
age_group    category
week            int64
quarter         int64
dtype: object

X_test dtypes:
job            object
marital        object
education      object
default         int64
balance         int64
housing         int64
loan            int64
duration        int64
campaign        int64
age_group    category
week            int64
quarter         int64
dtype: object


In [36]:
# Numerical columns
numerical_features = [
    'balance',
    'duration',
    'campaign'
]

# Categorical columns
categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan',
    'age_group',
    'week',
    'quarter'
]

# Already numeric - keep as they are
passthrough_features = [
    "default",
    "housing",
    "loan",
    "week",
    "quarter"
]


In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        
        # Scale numerical features
        ("num", StandardScaler(), numerical_features),
        
        # One-Hot Encode categorical features
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        
        # Keep already numeric features unchanged
        ("pass", "passthrough", passthrough_features)
    ]
)

## Apply Preprocessing

In [38]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Before preprocessing:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nAfter preprocessing:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

Before preprocessing:
X_train: (30000, 12)
X_test : (10000, 12)

After preprocessing:
X_train_processed: (30000, 46)
X_test_processed : (10000, 46)


In [ ]:
# Check Processed Feature Names

feature_names = preprocessor.get_feature_names_out()

print("Total Processed Features:", len(feature_names))

for i, feature in enumerate(feature_names, start=1):
    print(i, feature)

## Regular Random Forest - Model 1

In [45]:
regular_rf_1 = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

regular_rf_1.fit(
    X_train_processed,
    y_train
)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [46]:
y_pred_regular_rf_1 = regular_rf_1.predict(
    X_test_processed
)

y_prob_regular_rf_1 = regular_rf_1.predict_proba(
    X_test_processed
)[:, 1]

In [48]:
## Evaluation - Regular RF 1

In [47]:
accuracy_regular_rf_1 = accuracy_score(
    y_test,
    y_pred_regular_rf_1
)

precision_regular_rf_1 = precision_score(
    y_test,
    y_pred_regular_rf_1
)

recall_regular_rf_1 = recall_score(
    y_test,
    y_pred_regular_rf_1
)

f1_regular_rf_1 = f1_score(
    y_test,
    y_pred_regular_rf_1
)

roc_auc_regular_rf_1 = roc_auc_score(
    y_test,
    y_prob_regular_rf_1
)

cm_regular_rf_1 = confusion_matrix(
    y_test,
    y_pred_regular_rf_1
)

print("Regular Random Forest - Model 1")
print("--------------------------------")
print(f"Accuracy : {accuracy_regular_rf_1:.4f}")
print(f"Precision: {precision_regular_rf_1:.4f}")
print(f"Recall   : {recall_regular_rf_1:.4f}")
print(f"F1 Score : {f1_regular_rf_1:.4f}")
print(f"ROC-AUC  : {roc_auc_regular_rf_1:.4f}")

print("\nConfusion Matrix:")
print(cm_regular_rf_1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_regular_rf_1
    )
)

Regular Random Forest - Model 1
--------------------------------
Accuracy : 0.9316
Precision: 0.5680
Recall   : 0.2307
F1 Score : 0.3281
ROC-AUC  : 0.9042

Confusion Matrix:
[[9149  127]
 [ 557  167]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96      9276
           1       0.57      0.23      0.33       724

    accuracy                           0.93     10000
   macro avg       0.76      0.61      0.65     10000
weighted avg       0.92      0.93      0.92     10000



## GridSearchCV - Run 1

In [49]:
rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [51]:
param_grid_1 = {
    "n_estimators": [75, 100, 150, 200],
    "criterion": ["gini", "entropy"],
    "max_depth": [10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt"]
}

grid_search_1 = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_1,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_1.fit(
    X_train_processed,
    y_train
)

print("Best Parameters - Grid Search 1:")
print(grid_search_1.best_params_)

print("\nBest CV F1 Score:")
print(f"{grid_search_1.best_score_:.4f}")

Fitting 5 folds for each of 360 candidates, totalling 1800 fits
Best Parameters - Grid Search 1:
{'criterion': 'entropy', 'max_depth': 25, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}

Best CV F1 Score:
0.5196


## Test Evaluation - Best RF 1

In [54]:
best_rf_1 = grid_search_1.best_estimator_

print("Best RF - Run 1:")
print(best_rf_1)

Best RF - Run 1:
RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=25, min_samples_leaf=2, min_samples_split=10,
                       n_estimators=200, n_jobs=-1, random_state=42)


In [52]:
best_rf_1 = grid_search_1.best_estimator_

y_pred_best_rf_1 = best_rf_1.predict(
    X_test_processed
)

y_prob_best_rf_1 = best_rf_1.predict_proba(
    X_test_processed
)[:, 1]

accuracy_best_rf_1 = accuracy_score(
    y_test,
    y_pred_best_rf_1
)

precision_best_rf_1 = precision_score(
    y_test,
    y_pred_best_rf_1
)

recall_best_rf_1 = recall_score(
    y_test,
    y_pred_best_rf_1
)

f1_best_rf_1 = f1_score(
    y_test,
    y_pred_best_rf_1
)

roc_auc_best_rf_1 = roc_auc_score(
    y_test,
    y_prob_best_rf_1
)

cm_best_rf_1 = confusion_matrix(
    y_test,
    y_pred_best_rf_1
)

print("Grid Search 1 - Best Random Forest")
print("-----------------------------------")
print(f"Accuracy : {accuracy_best_rf_1:.4f}")
print(f"Precision: {precision_best_rf_1:.4f}")
print(f"Recall   : {recall_best_rf_1:.4f}")
print(f"F1 Score : {f1_best_rf_1:.4f}")
print(f"ROC-AUC  : {roc_auc_best_rf_1:.4f}")

print("\nConfusion Matrix:")
print(cm_best_rf_1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_best_rf_1
    )
)

Grid Search 1 - Best Random Forest
-----------------------------------
Accuracy : 0.9126
Precision: 0.4324
Recall   : 0.6630
F1 Score : 0.5234
ROC-AUC  : 0.9175

Confusion Matrix:
[[8646  630]
 [ 244  480]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.93      0.95      9276
           1       0.43      0.66      0.52       724

    accuracy                           0.91     10000
   macro avg       0.70      0.80      0.74     10000
weighted avg       0.93      0.91      0.92     10000



## GridSearchCV - Run 2

In [55]:
param_grid_2 = {
    "n_estimators": [150, 200, 250, 300],
    "criterion": ["gini", "entropy"],
    "max_depth": [20, 23, 27, 30],
    "min_samples_split": [5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 3, 4],
    "max_features": ["sqrt"]
}

grid_search_2 = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_2,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_2.fit(
    X_train_processed,
    y_train
)

print("Best Parameters - Grid Search 2:")
print(grid_search_2.best_params_)

print("\nBest CV F1 Score:")
print(f"{grid_search_2.best_score_:.4f}")

Fitting 5 folds for each of 512 candidates, totalling 2560 fits
Best Parameters - Grid Search 2:
{'criterion': 'entropy', 'max_depth': 23, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 15, 'n_estimators': 150}

Best CV F1 Score:
0.5238


In [56]:
best_rf_2 = grid_search_2.best_estimator_

print("Best RF - Run 2:")
print(best_rf_2)

Best RF - Run 2:
RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=23, min_samples_split=15, n_estimators=150,
                       n_jobs=-1, random_state=42)


In [57]:
y_pred_best_rf_2 = best_rf_2.predict(X_test_processed)
y_prob_best_rf_2 = best_rf_2.predict_proba(X_test_processed)[:, 1]

accuracy_best_rf_2 = accuracy_score(y_test, y_pred_best_rf_2)
precision_best_rf_2 = precision_score(y_test, y_pred_best_rf_2)
recall_best_rf_2 = recall_score(y_test, y_pred_best_rf_2)
f1_best_rf_2 = f1_score(y_test, y_pred_best_rf_2)
roc_auc_best_rf_2 = roc_auc_score(y_test, y_prob_best_rf_2)

cm_best_rf_2 = confusion_matrix(y_test, y_pred_best_rf_2)

print("Grid Search 2 - Best Random Forest")
print(f"Accuracy : {accuracy_best_rf_2:.4f}")
print(f"Precision: {precision_best_rf_2:.4f}")
print(f"Recall   : {recall_best_rf_2:.4f}")
print(f"F1 Score : {f1_best_rf_2:.4f}")
print(f"ROC-AUC  : {roc_auc_best_rf_2:.4f}")

print("\nConfusion Matrix:")
print(cm_best_rf_2)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf_2))

Grid Search 2 - Best Random Forest
Accuracy : 0.9096
Precision: 0.4224
Recall   : 0.6768
F1 Score : 0.5202
ROC-AUC  : 0.9169

Confusion Matrix:
[[8606  670]
 [ 234  490]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.93      0.95      9276
           1       0.42      0.68      0.52       724

    accuracy                           0.91     10000
   macro avg       0.70      0.80      0.74     10000
weighted avg       0.93      0.91      0.92     10000



## GridSearchCV - Run 3

In [58]:
param_grid_3 = {
    "n_estimators": [125, 150, 175, 200, 225],
    "criterion": ["entropy"],
    "max_depth": [21, 23, 25, 27, 29],
    "min_samples_split": [8, 10, 12, 15],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt"]
}

grid_search_3 = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_3,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search_3.fit(
    X_train_processed,
    y_train
)

print("Best Parameters - Grid Search 3:")
print(grid_search_3.best_params_)

print("\nBest CV F1 Score:")
print(f"{grid_search_3.best_score_:.4f}")

Fitting 5 folds for each of 300 candidates, totalling 1500 fits
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=125; total time=   7.0s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=150; total time=   8.7s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=150; total time=   8.7s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=175; total time=  10.1s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=200; total time=  11.1s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=225; total time=  12.6s
[CV] END criterion=entropy, max_depth=21, max_features=sqrt, min_samples_leaf=1, min_samples_split=10,

In [59]:
best_rf_3 = grid_search_3.best_estimator_

print("Best RF - Run 3:")
print(best_rf_3)

Best RF - Run 3:
RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=25, min_samples_split=15, n_estimators=150,
                       n_jobs=-1, random_state=42)


In [60]:
best_rf_3 = grid_search_3.best_estimator_

y_pred_best_rf_3 = best_rf_3.predict(X_test_processed)
y_prob_best_rf_3 = best_rf_3.predict_proba(X_test_processed)[:, 1]

accuracy_best_rf_3 = accuracy_score(y_test, y_pred_best_rf_3)
precision_best_rf_3 = precision_score(y_test, y_pred_best_rf_3)
recall_best_rf_3 = recall_score(y_test, y_pred_best_rf_3)
f1_best_rf_3 = f1_score(y_test, y_pred_best_rf_3)
roc_auc_best_rf_3 = roc_auc_score(y_test, y_prob_best_rf_3)

cm_best_rf_3 = confusion_matrix(y_test, y_pred_best_rf_3)

print("Grid Search 3 - Best Random Forest")
print(f"Accuracy : {accuracy_best_rf_3:.4f}")
print(f"Precision: {precision_best_rf_3:.4f}")
print(f"Recall   : {recall_best_rf_3:.4f}")
print(f"F1 Score : {f1_best_rf_3:.4f}")
print(f"ROC-AUC  : {roc_auc_best_rf_3:.4f}")

print("\nConfusion Matrix:")
print(cm_best_rf_3)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf_3))

Grid Search 3 - Best Random Forest
Accuracy : 0.9104
Precision: 0.4238
Recall   : 0.6602
F1 Score : 0.5162
ROC-AUC  : 0.9174

Confusion Matrix:
[[8626  650]
 [ 246  478]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.93      0.95      9276
           1       0.42      0.66      0.52       724

    accuracy                           0.91     10000
   macro avg       0.70      0.80      0.73     10000
weighted avg       0.93      0.91      0.92     10000

[CV] END criterion=entropy, max_depth=27, max_features=sqrt, min_samples_leaf=1, min_samples_split=8, n_estimators=225; total time=  11.9s
[CV] END criterion=entropy, max_depth=27, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=125; total time=   6.4s
[CV] END criterion=entropy, max_depth=27, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=150; total time=   7.7s
[CV] END criterion=entropy, max_depth=27, max_features=sqrt

In [61]:
from imblearn.over_sampling import SMOTE

In [62]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nPercentage:")
print(y_train.value_counts(normalize=True) * 100)

Before SMOTE:
y
0    27828
1     2172
Name: count, dtype: int64

Percentage:
y
0    92.76
1     7.24
Name: proportion, dtype: float64


In [63]:
smote_1 = SMOTE(
    sampling_strategy=0.5,
    random_state=42
)

X_train_smote_1, y_train_smote_1 = smote_1.fit_resample(
    X_train_processed,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote_1.value_counts())

print("\nOriginal Training Shape:")
print(X_train_processed.shape)

print("\nSMOTE Training Shape:")
print(X_train_smote_1.shape)

Before SMOTE:
y
0    27828
1     2172
Name: count, dtype: int64

After SMOTE:
y
0    27828
1    13914
Name: count, dtype: int64

Original Training Shape:
(30000, 46)

SMOTE Training Shape:
(41742, 46)


## Random Forest on SMOTE data

In [64]:
smote_rf_1 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

smote_rf_1.fit(
    X_train_smote_1,
    y_train_smote_1
)

y_pred_smote_rf_1 = smote_rf_1.predict(X_test_processed)
y_prob_smote_rf_1 = smote_rf_1.predict_proba(X_test_processed)[:, 1]

accuracy_smote_rf_1 = accuracy_score(y_test, y_pred_smote_rf_1)
precision_smote_rf_1 = precision_score(y_test, y_pred_smote_rf_1)
recall_smote_rf_1 = recall_score(y_test, y_pred_smote_rf_1)
f1_smote_rf_1 = f1_score(y_test, y_pred_smote_rf_1)
roc_auc_smote_rf_1 = roc_auc_score(y_test, y_prob_smote_rf_1)

print("SMOTE Random Forest - Model 1")
print(f"Accuracy : {accuracy_smote_rf_1:.4f}")
print(f"Precision: {precision_smote_rf_1:.4f}")
print(f"Recall   : {recall_smote_rf_1:.4f}")
print(f"F1 Score : {f1_smote_rf_1:.4f}")
print(f"ROC-AUC  : {roc_auc_smote_rf_1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote_rf_1))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote_rf_1))

SMOTE Random Forest - Model 1
Accuracy : 0.9273
Precision: 0.4971
Recall   : 0.3536
F1 Score : 0.4132
ROC-AUC  : 0.9080

Confusion Matrix:
[[9017  259]
 [ 468  256]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      9276
           1       0.50      0.35      0.41       724

    accuracy                           0.93     10000
   macro avg       0.72      0.66      0.69     10000
weighted avg       0.92      0.93      0.92     10000



## SMOTE ratio comparison

In [65]:
smote_ratios = [0.5, 0.75, 1.0]

smote_results = []

for ratio in smote_ratios:

    smote = SMOTE(
        sampling_strategy=ratio,
        random_state=42
    )

    X_train_smote, y_train_smote = smote.fit_resample(
        X_train_processed,
        y_train
    )

    smote_rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    smote_rf.fit(X_train_smote, y_train_smote)

    y_pred = smote_rf.predict(X_test_processed)
    y_prob = smote_rf.predict_proba(X_test_processed)[:, 1]

    smote_results.append({
        "SMOTE Ratio": ratio,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Predicted Yes": y_pred.sum()
    })

smote_results_df = pd.DataFrame(smote_results)

smote_results_df

,SMOTE Ratio,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Predicted Yes
0,0.50,0.9273,0.497087,0.353591,0.413236,0.907977,515
1,0.75,0.9242,0.469091,0.356354,0.405024,0.908246,550
2,1.00,0.9229,0.457961,0.353591,0.399065,0.909595,559


In [66]:
smote_rf_2 = RandomForestClassifier(
    n_estimators=200,
    criterion="entropy",
    max_depth=25,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

smote_rf_2.fit(
    X_train_smote_1,
    y_train_smote_1
)

y_pred_smote_rf_2 = smote_rf_2.predict(X_test_processed)
y_prob_smote_rf_2 = smote_rf_2.predict_proba(X_test_processed)[:, 1]

accuracy_smote_rf_2 = accuracy_score(y_test, y_pred_smote_rf_2)
precision_smote_rf_2 = precision_score(y_test, y_pred_smote_rf_2)
recall_smote_rf_2 = recall_score(y_test, y_pred_smote_rf_2)
f1_smote_rf_2 = f1_score(y_test, y_pred_smote_rf_2)
roc_auc_smote_rf_2 = roc_auc_score(y_test, y_prob_smote_rf_2)

print("SMOTE Random Forest - Model 2")
print(f"Accuracy : {accuracy_smote_rf_2:.4f}")
print(f"Precision: {precision_smote_rf_2:.4f}")
print(f"Recall   : {recall_smote_rf_2:.4f}")
print(f"F1 Score : {f1_smote_rf_2:.4f}")
print(f"ROC-AUC  : {roc_auc_smote_rf_2:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote_rf_2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote_rf_2))

SMOTE Random Forest - Model 2
Accuracy : 0.9247
Precision: 0.4771
Recall   : 0.4171
F1 Score : 0.4451
ROC-AUC  : 0.9152

Confusion Matrix:
[[8945  331]
 [ 422  302]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      9276
           1       0.48      0.42      0.45       724

    accuracy                           0.92     10000
   macro avg       0.72      0.69      0.70     10000
weighted avg       0.92      0.92      0.92     10000



## GridSearchCV - Run 4

In [2]:
from sklearn.model_selection import GridSearchCV

In [3]:
param_grid_4 = {
    "n_estimators": [125, 150, 175, 200],
    "criterion": ["entropy"],
    "max_depth": [23, 25, 27],
    "min_samples_split": [12, 15, 18, 20],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2", 0.5]
}

grid_search_4 = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_4,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_4.fit(
    X_train_processed,
    y_train
)

print("Best Parameters - Grid Search 4:")
print(grid_search_4.best_params_)

print("\nBest CV F1 Score:")
print(f"{grid_search_4.best_score_:.4f}")

NameError: name 'rf' is not defined